In [1]:
!rm -rf /kaggle/working/*

In [ ]:
!unzip -q /kaggle/input/notebooks/abhishekkujur19/libriphrase-data/_output_.zip -d /kaggle/working/

In [ ]:
!rm /kaggle/working/libriphrase_dataset.zip

In [ ]:
!unzip -q /kaggle/input/notebooks/abhishekkujur19/agents-data/coqui_synthetic_cleaned.zip -d /kaggle/working/

In [ ]:
!mkdir -p /kaggle/working/coqui_dataset

# 2. Move the folders and files into it
!mv /kaggle/working/hard_negatives /kaggle/working/coqui_dataset/
!mv /kaggle/working/manifest.csv /kaggle/working/coqui_dataset/
!mv /kaggle/working/wake_words /kaggle/working/coqui_dataset/

In [ ]:
# 1. Install Cython first — required to compile some of NeMo's fast audio backends
!pip install --quiet Cython

# 2. Install SpeechBrain cleanly without changing your core PyTorch system wheels
!pip install --quiet speechbrain

# 3. Install the OPTIMIZED targeted ASR slice of NVIDIA NeMo 
# Avoid using [all] as it will crash your container's memory allocation limits
!pip install --quiet nemo_toolkit[asr]

# 4. Install standard soundfile processing systems for backend data loading
!pip install --quiet soundfile

******MANIFEST******

In [7]:
import os
import json
import pandas as pd
import glob
import random
from collections import Counter

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
COQUI_DIR       = "/kaggle/working/coqui_dataset"
GSC_DIR         = "/kaggle/input/datasets/sylkaladin/speech-commands-v2"
LIBRIPHRASE_DIR = "/kaggle/working/libriphrase_raw"
OUTPUT_DIR      = "/kaggle/working/pcov-kws-manifests"

os.makedirs(OUTPUT_DIR, exist_ok=True)
TRAIN_JSON_PATH = os.path.join(OUTPUT_DIR, 'train_manifest.json')
VAL_JSON_PATH   = os.path.join(OUTPUT_DIR, 'val_manifest.json')

train_manifest = []
val_manifest   = []
random.seed(42)

# ─────────────────────────────────────────────
# TARGETS — tune these knobs
# ─────────────────────────────────────────────
TARGET_UNKNOWN_TRAIN = 40_000   # cap GSC unknown
TARGET_UNKNOWN_VAL   = 10_000
TARGET_SILENCE_TRAIN = 8_000    # silence crops (real diversity via chunking)
TARGET_SILENCE_VAL   = 2_000
LIBRI_CAP_TRAIN      = 20_000   # cap LibriPhrase unknown
LIBRI_CAP_VAL        = 5_000


# ─────────────────────────────────────────────
# 1. COQUI SYNTHETIC (Classes 0 & 3) — unchanged logic, just report
# ─────────────────────────────────────────────
print("--- TASK 1: COQUI SYNTHETIC DATA ---")

coqui_csv = os.path.join(COQUI_DIR, 'manifest.csv')
coqui_df  = pd.read_csv(coqui_csv)

all_speakers = coqui_df['speaker'].unique().tolist()
random.shuffle(all_speakers)
split_idx      = int(len(all_speakers) * 0.8)
train_speakers = set(all_speakers[:split_idx])
val_speakers   = set(all_speakers[split_idx:])

print(f"Speakers — Train: {len(train_speakers)} | Val: {len(val_speakers)}")

ww_train, ww_val, hn_train, hn_val = 0, 0, 0, 0
for _, row in coqui_df.iterrows():
    try:
        spk_id = int(str(row['speaker']).replace('p', ''))
    except (ValueError, AttributeError):
        spk_id = -1

    sample = {
        'path':      os.path.join(COQUI_DIR, row['filename']),
        'kws_label': int(row['label']),
        'spk_label': spk_id
    }
    if row['speaker'] in train_speakers:
        train_manifest.append(sample)
        if int(row['label']) == 0: ww_train += 1
        else: hn_train += 1
    else:
        val_manifest.append(sample)
        if int(row['label']) == 0: ww_val += 1
        else: hn_val += 1

print(f"Wake Word  — Train: {ww_train:,} | Val: {ww_val:,}")
print(f"Hard-Neg   — Train: {hn_train:,} | Val: {hn_val:,}")


# ─────────────────────────────────────────────
# 2. LIBRIPHRASE UNKNOWN (Class 2) — CAPPED
# ─────────────────────────────────────────────
print("\n--- TASK 2: LIBRIPHRASE UNKNOWN ---")

libri_wavs = glob.glob(os.path.join(LIBRIPHRASE_DIR, "**/*.wav"), recursive=True)
if libri_wavs:
    random.shuffle(libri_wavs)
    # Cap total before splitting so val cap is proportional
    libri_wavs = libri_wavs[:LIBRI_CAP_TRAIN + LIBRI_CAP_VAL]
    libri_split = int(len(libri_wavs) * 0.8)

    for wav_path in libri_wavs[:libri_split]:
        train_manifest.append({'path': wav_path, 'kws_label': 2, 'spk_label': -1})
    for wav_path in libri_wavs[libri_split:]:
        val_manifest.append({'path': wav_path, 'kws_label': 2, 'spk_label': -1})

    print(f"Train: {libri_split:,} | Val: {len(libri_wavs)-libri_split:,}")
else:
    print("⚠️ No LibriPhrase files found.")


# ─────────────────────────────────────────────
# 3. GSC V2 — CAPPED UNKNOWN + CHUNKED SILENCE
# ─────────────────────────────────────────────
print("\n--- TASK 3: GOOGLE SPEECH COMMANDS V2 ---")

if os.path.exists(GSC_DIR):
    all_gsc = glob.glob(os.path.join(GSC_DIR, "**/*.wav"), recursive=True)
    random.shuffle(all_gsc)

    silence_files = [p for p in all_gsc
                     if "_background_noise_" in p or "silence" in p.lower()]
    word_files    = [p for p in all_gsc
                     if "_background_noise_" not in p and "silence" not in p.lower()]

    # ── Unknown words: cap total, then 80/20 split ──
    word_files = word_files[:TARGET_UNKNOWN_TRAIN + TARGET_UNKNOWN_VAL]
    word_split = int(len(word_files) * 0.8)
    for p in word_files[:word_split]:
        train_manifest.append({'path': p, 'kws_label': 2, 'spk_label': -1})
    for p in word_files[word_split:]:
        val_manifest.append({'path': p, 'kws_label': 2, 'spk_label': -1})

    # ── Silence: chunk each file into non-overlapping 1s windows ──
    # This gives REAL diversity instead of 2000 copies of the same file
    SAMPLE_RATE   = 16000

# make_manifest.py — replace the silence chunking section only

    silence_chunks_train = []
    silence_chunks_val   = []

    import soundfile as sf_probe
    for sil_path in silence_files:
        try:
            info       = sf_probe.info(sil_path)
            total_secs = info.duration
            n_chunks   = int(total_secs)
            for chunk_idx in range(n_chunks):
                entry = {
                    'path':      sil_path,
                    'kws_label': 1,
                    'spk_label': -1,
                    'chunk_idx': chunk_idx
                }
                if random.random() < 0.8:
                    silence_chunks_train.append(entry)
                else:
                    silence_chunks_val.append(entry)
        except Exception:
            continue

    # ── If not enough silence chunks, repeat with random chunk_idx offsets ──
    if len(silence_chunks_train) < TARGET_SILENCE_TRAIN and silence_files:
        print(f"⚠️  Only {len(silence_chunks_train)} silence chunks — padding via random re-crops...")
        while len(silence_chunks_train) < TARGET_SILENCE_TRAIN:
            sil_path = random.choice(silence_files)
            try:
                info    = sf_probe.info(sil_path)
                n_chunks = max(1, int(info.duration))
                silence_chunks_train.append({
                    'path':      sil_path,
                    'kws_label': 1,
                    'spk_label': -1,
                    'chunk_idx': random.randint(0, n_chunks - 1)
                })
            except Exception:
                continue

    if len(silence_chunks_val) < TARGET_SILENCE_VAL and silence_files:
        while len(silence_chunks_val) < TARGET_SILENCE_VAL:
            sil_path = random.choice(silence_files)
            try:
                info    = sf_probe.info(sil_path)
                n_chunks = max(1, int(info.duration))
                silence_chunks_val.append({
                    'path':      sil_path,
                    'kws_label': 1,
                    'spk_label': -1,
                    'chunk_idx': random.randint(0, n_chunks - 1)
                })
            except Exception:
                continue

    # Cap to targets
    random.shuffle(silence_chunks_train)
    random.shuffle(silence_chunks_val)
    silence_chunks_train = silence_chunks_train[:TARGET_SILENCE_TRAIN]
    silence_chunks_val   = silence_chunks_val[:TARGET_SILENCE_VAL]

    train_manifest.extend(silence_chunks_train)
    val_manifest.extend(silence_chunks_val)

    print(f"Unknown  — Train: {word_split:,} | Val: {len(word_files)-word_split:,}")
    print(f"Silence  — Train: {len(silence_chunks_train):,} | Val: {len(silence_chunks_val):,}")
else:
    print("⚠️ GSC directory not found.")


# ─────────────────────────────────────────────
# 4. SHUFFLE + EXPORT
# ─────────────────────────────────────────────
random.shuffle(train_manifest)
random.shuffle(val_manifest)

with open(TRAIN_JSON_PATH, 'w') as f:
    json.dump(train_manifest, f, indent=4)
with open(VAL_JSON_PATH, 'w') as f:
    json.dump(val_manifest, f, indent=4)


# ─────────────────────────────────────────────
# 5. DISTRIBUTION REPORT
# ─────────────────────────────────────────────
CLASS_NAMES = {0: 'Wake Word', 1: 'Silence', 2: 'Unknown', 3: 'Hard-Negative'}
train_labels = Counter(s['kws_label'] for s in train_manifest)
val_labels   = Counter(s['kws_label'] for s in val_manifest)

print("\n--- CLASS DISTRIBUTION REPORT ---")
print(f"{'Class':<20} {'Train':>10} {'Train%':>8} {'Val':>10} {'Val%':>8}")
print("-" * 60)
total_train = len(train_manifest)
total_val   = len(val_manifest)
for lbl in sorted(CLASS_NAMES):
    tr = train_labels.get(lbl, 0)
    vl = val_labels.get(lbl, 0)
    print(f"{CLASS_NAMES[lbl]:<20} {tr:>10,} {100*tr/total_train:>7.1f}% "
          f"{vl:>10,} {100*vl/total_val:>7.1f}%")
print("-" * 60)
print(f"{'TOTAL':<20} {total_train:>10,} {'100.0%':>8} {total_val:>10,} {'100.0%':>8}")

# Imbalance warning
max_tr = max(train_labels.values())
min_tr = min(train_labels.values())
ratio  = max_tr / max(min_tr, 1)
if ratio > 10:
    print(f"\n⚠️  WARNING: Max/min class ratio = {ratio:.1f}x — consider generating more Hard-Neg/Wake Word data")
else:
    print(f"\n✅ Class ratio = {ratio:.1f}x — acceptable for weighted CE loss")


--- TASK 1: COQUI SYNTHETIC DATA ---
Speakers — Train: 87 | Val: 22
Wake Word  — Train: 35,120 | Val: 8,880
Hard-Neg   — Train: 11,973 | Val: 3,027

--- TASK 2: LIBRIPHRASE UNKNOWN ---
Train: 20,000 | Val: 5,000

--- TASK 3: GOOGLE SPEECH COMMANDS V2 ---
⚠️  Only 303 silence chunks — padding via random re-crops...
Unknown  — Train: 40,000 | Val: 10,000
Silence  — Train: 8,000 | Val: 2,000

--- CLASS DISTRIBUTION REPORT ---
Class                     Train   Train%        Val     Val%
------------------------------------------------------------
Wake Word                35,120    30.5%      8,880    30.7%
Silence                   8,000     7.0%      2,000     6.9%
Unknown                  60,000    52.1%     15,000    51.9%
Hard-Negative            11,973    10.4%      3,027    10.5%
------------------------------------------------------------
TOTAL                   115,093   100.0%     28,907   100.0%

✅ Class ratio = 7.5x — acceptable for weighted CE loss


******DATASET******

In [8]:
%%writefile dataset.py
import os
import random
import glob
import torch
import torchaudio.functional as F_audio
from torch.utils.data import Dataset
import soundfile as sf
from tqdm import tqdm

def load_audio_safe(file_path):
    clean_path = str(file_path).replace('\n', '').replace('\r', '').strip()
    wav, sr = sf.read(clean_path)
    waveform = torch.from_numpy(wav).float()
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.t()
    return waveform, sr


# ─────────────────────────────────────────────
# 1. RAM-CACHED RIR (Zero Disk I/O)
# ─────────────────────────────────────────────
class RAMCachedRIR:
    def __init__(self, rir_dir, sample_rate=16000, max_cache=2000):
        self.sample_rate = sample_rate
        all_files = glob.glob(os.path.join(rir_dir, '**', '*.wav'), recursive=True)
        self.cache = []

        if not all_files:
            print("⚠️ No RIR files found. Distance simulation will be bypassed.")
            return

        random.shuffle(all_files)
        selected_files = all_files[:max_cache]

        print(f"📥 Caching {len(selected_files)} RIR files into RAM...")
        for path in tqdm(selected_files, desc="Loading RIRs", leave=False):
            wav, sr = load_audio_safe(path)
            if sr != sample_rate:
                wav = F_audio.resample(wav, sr, sample_rate)
            if wav.shape[0] > 1:
                wav = torch.mean(wav, dim=0, keepdim=True)
            max_len = sample_rate * 2
            if wav.shape[1] > max_len:
                wav = wav[:, :max_len]
            norm = torch.norm(wav, p=2)
            if norm > 0:
                wav = wav / norm
            self.cache.append(wav)

    def __call__(self, clean_waveform):
        if not self.cache:
            return clean_waveform
        rir_waveform = random.choice(self.cache)
        augmented = F_audio.fftconvolve(clean_waveform, rir_waveform)
        return torch.clamp(augmented[:, :clean_waveform.shape[1]], min=-1.0, max=1.0)


# ─────────────────────────────────────────────
# 2. RAM-CACHED MUSAN (Zero Disk I/O)
# ─────────────────────────────────────────────
class RAMCachedMUSAN:
    def __init__(self, musan_dir, sample_rate=16000, max_cache=2000):
        self.sample_rate = sample_rate
        all_files = glob.glob(os.path.join(musan_dir, '**', '*.wav'), recursive=True)
        self.cache = []

        if not all_files:
            print("⚠️ No MUSAN files found. Noise augmentation will be bypassed.")
            return

        random.shuffle(all_files)
        selected_files = all_files[:max_cache]

        print(f"📥 Caching {len(selected_files)} MUSAN files into RAM...")
        for path in tqdm(selected_files, desc="Loading MUSAN", leave=False):
            wav, sr = load_audio_safe(path)
            if sr != sample_rate:
                wav = F_audio.resample(wav, sr, sample_rate)
            if wav.shape[0] > 1:
                wav = torch.mean(wav, dim=0, keepdim=True)
            max_len = sample_rate * 5
            if wav.shape[1] > max_len:
                start = random.randint(0, wav.shape[1] - max_len)
                wav = wav[:, start:start+max_len]
            self.cache.append(wav)

    def __call__(self, clean_waveform, target_snr_db=-5.0):
        if not self.cache:
            return clean_waveform
        noise_waveform = random.choice(self.cache)
        if noise_waveform.shape[1] > clean_waveform.shape[1]:
            start = random.randint(0, noise_waveform.shape[1] - clean_waveform.shape[1])
            noise_waveform = noise_waveform[:, start:start + clean_waveform.shape[1]]
        else:
            pad_amount = clean_waveform.shape[1] - noise_waveform.shape[1]
            noise_waveform = torch.nn.functional.pad(noise_waveform, (0, pad_amount))
        clean_power = torch.mean(clean_waveform ** 2)
        noise_power = torch.mean(noise_waveform ** 2)
        if clean_power == 0 or noise_power == 0:
            return clean_waveform
        k = torch.sqrt((clean_power / noise_power) * (10 ** (-target_snr_db / 10)))
        noisy = clean_waveform + k * noise_waveform
        return torch.clamp(noisy, min=-1.0, max=1.0)


# ─────────────────────────────────────────────
# 3. THE MASTER MULTI-TASK DATASET
# ─────────────────────────────────────────────
class PCOVMultiTaskDataset(Dataset):
    def __init__(self, data_manifest, musan_dir=None, rir_dir=None,
                 sample_rate=16000, duration_sec=1.0):
        self.data_list     = data_manifest
        self.sample_rate   = sample_rate
        self.target_length = int(sample_rate * duration_sec)
        self.rir_augmenter   = RAMCachedRIR(rir_dir, sample_rate)     if rir_dir   else None
        self.musan_augmenter = RAMCachedMUSAN(musan_dir, sample_rate) if musan_dir else None

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        waveform, sr = load_audio_safe(item['path'])

        # ── Resample + mono FIRST ──
        if sr != self.sample_rate:
            waveform = F_audio.resample(waveform, sr, self.sample_rate)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # ── NaN/silence guard — local vars only, never mutate item dict ──
        if torch.isnan(waveform).any() or torch.isinf(waveform).any() \
                or waveform.pow(2).mean().sqrt() < 1e-8:
            waveform  = torch.zeros(1, self.target_length)
            kws_label = 1   # silence
            spk_label = -1
        else:
            kws_label = item['kws_label']
            spk_label = item.get('spk_label', -1)

        # ── Crop / pad ──
        current_len = waveform.shape[1]
        chunk_idx   = item.get('chunk_idx', None)

        if chunk_idx is not None:
            start = chunk_idx * self.target_length
            end   = start + self.target_length
            if end <= current_len:
                waveform = waveform[:, start:end]
            else:
                waveform = waveform[:, start:] if start < current_len \
                           else torch.zeros(1, self.target_length)
                waveform = torch.nn.functional.pad(
                    waveform, (0, self.target_length - waveform.shape[1])
                )
        elif current_len > self.target_length:
            if kws_label in [0, 3]:
                start = 0
            else:
                start = random.randint(0, current_len - self.target_length)
            waveform = waveform[:, start:start + self.target_length]
        elif current_len < self.target_length:
            waveform = torch.nn.functional.pad(
                waveform, (0, self.target_length - current_len)
            )

        # ── Augmentations ──
        if self.rir_augmenter and random.random() < 0.30:
            waveform = self.rir_augmenter(waveform)
        if self.musan_augmenter and random.random() < 0.50:
            snr = random.uniform(-5.0, 0.0) if random.random() < 0.40 \
                  else random.uniform(0.0, 10.0)
            waveform = self.musan_augmenter(waveform, target_snr_db=snr)

        peak = waveform.abs().max()
        if peak > 1e-4:
            waveform = waveform / peak * 0.9

        return (
            waveform,
            torch.tensor(kws_label, dtype=torch.long),
            torch.tensor(spk_label, dtype=torch.long),
        )


Writing dataset.py


******FRONTEND******

In [9]:
%%writefile frontend.py
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import math


# ─────────────────────────────────────────────
#  DSP PRE-PROCESSING
# Note: As per architecture, WebRTC VAD and RNNoise are executed 
# on the C++ DSP layer prior to this PyTorch pipeline to save power.
# ─────────────────────────────────────────────


class PreEmphasis(nn.Module):
    """
    Boosts high frequencies to balance the spectrum of the human voice.
    Matches the 'Stage 1: Pre-emphasis' claim in the PPT.
    """
    def __init__(self, coef=0.97):
        super().__init__()
        self.coef = coef
        # FIR filter weight: y[t] = x[t] - coef * x[t-1]
        self.register_buffer('weight', torch.tensor([-self.coef, 1.0]).view(1, 1, 2))

    def forward(self, x):
        # x: (Batch, 1, Time)
        x_padded = F.pad(x, (1, 0), 'reflect')
        return F.conv1d(x_padded, self.weight)
    
    
# ─────────────────────────────────────────────
#  PCEN FEATURE EXTRACTION
# ─────────────────────────────────────────────

class PCENTransform(nn.Module):
    """
    Per-Channel Energy Normalization (Wang et al., ICASSP 2017).
    [UPGRADED] Parameters are constrained via Sigmoid/Softplus to prevent NaN crashes.
    """
    def __init__(self, gain=0.98, bias=2.0, power=0.5,
                 time_constant=0.025, eps=1e-6):
        super().__init__()
        self.eps = eps

        # ---------------------------------------------------------
        # INITIALIZATION USING INVERSE MATH
        # We use Inverse-Sigmoid (Logit) and Inverse-Softplus so that 
        # the parameters start EXACTLY at the default values.
        # ---------------------------------------------------------
        
        # Sigmoid forces alpha, r, and s to stay safely between (0.0, 1.0)
        # logit(x) = ln(x / (1 - x))
        self.raw_alpha = nn.Parameter(torch.tensor(math.log(gain / (1.0 - gain))))
        self.raw_r     = nn.Parameter(torch.tensor(math.log(power / (1.0 - power))))
        self.raw_s     = nn.Parameter(torch.tensor(math.log(time_constant / (1.0 - time_constant))))
        
        # Softplus forces delta to stay strictly positive (> 0.0)
        # inv_softplus(x) = ln(exp(x) - 1)
        self.raw_delta = nn.Parameter(torch.tensor(math.log(math.exp(bias) - 1.0)))

    def forward(self, x):
        # 1. Apply safety activations to the raw learnable parameters
        alpha = torch.sigmoid(self.raw_alpha)
        r     = torch.sigmoid(self.raw_r)
        s     = torch.sigmoid(self.raw_s)
        delta = F.softplus(self.raw_delta)

        # 2. Standard PCEN computation using the safe variables
        # [PCGRAD FIX] list+stack instead of in-place writes
        # In-place m[...,t]= breaks retain_graph=True in PCGrad second backward.
        # [ONNX WARNING] Dynamic loop needs JIT script or 1D conv for export.py
        frames = [x[..., 0]]
        for t in range(1, x.shape[-1]):
            frames.append((1.0 - s) * frames[-1] + s * x[..., t])
        m = torch.stack(frames, dim=-1)
            
        return ((x / (self.eps + m) ** alpha) + delta) ** r - delta ** r


class PCENFrontend(nn.Module):
    """
    Stages 1+2: Raw waveform → 80-bin PCEN Mel-Spectrogram.
    Includes SpecAugment (Time/Frequency Masking) active ONLY during training.
    """
    def __init__(self, sample_rate=16000, n_mels=80):
        super().__init__()
        self.pre_emphasis = PreEmphasis(coef=0.97)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=400,        
            hop_length=160,   
            n_mels=n_mels,
            f_min=20,
            f_max=8000,
            power=2.0         # Explicit declaration per hackathon best-practices
        )
        self.pcen = PCENTransform()
        
        # --- SPECAUGMENT ---
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=20)

    def forward(self, waveform):
        x = self.pre_emphasis(waveform)  # FIR boost consonants before mel
        mel = self.mel(x.squeeze(dim=1))  # squeeze(dim=1) — safe for batch=1   
        pcen_out = self.pcen(mel)             
        
        if self.training:
            pcen_out = self.freq_mask(pcen_out)
            pcen_out = self.time_mask(pcen_out)

        return pcen_out.unsqueeze(1)          


Writing frontend.py


******MODEL******

In [10]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

# ─────────────────────────────────────────────
# ATTENTION & REGULARIZATION BLOCKS
# ─────────────────────────────────────────────

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation Block.
    CRITICAL FOR -5dB SNR: Dynamically recalibrates channel weights, 
    suppressing MUSAN/RIR noise and exciting human formants.
    """
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False)
        self.relu = nn.ReLU6(inplace=True)
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        self.hardsigmoid = nn.Hardsigmoid(inplace=True)

    def forward(self, x):
        scale = self.avg_pool(x)
        scale = self.fc1(scale)
        scale = self.relu(scale)
        scale = self.fc2(scale)
        scale = self.hardsigmoid(scale)
        return x * scale


# ─────────────────────────────────────────────
# DS-CNN BACKBONE WITH RESIDUALS
# ─────────────────────────────────────────────

class DSConvBlock(nn.Module):
    """
    Depthwise Separable Convolution Block + Residuals + SE Attention.
    Uses ReLU6 to bound activations to [0,6] — critical for NPU INT8 deployment.
    """
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.1):
        super().__init__()
        self.use_res = (in_channels == out_channels) and (stride == 1 or stride == (1, 1))

        self.dw = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=3, padding=1,
            stride=stride, groups=in_channels, bias=False
        )
        self.bn_dw = nn.BatchNorm2d(in_channels)
        self.act_dw = nn.ReLU6(inplace=True)

        self.pw = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1, bias=False
        )
        self.bn_pw = nn.BatchNorm2d(out_channels)
        
        self.se = SEBlock(out_channels)
        
        self.act_pw = nn.ReLU6(inplace=True)
        self.drop = nn.Dropout2d(p=dropout_p)

    def forward(self, x):
        identity = x
        
        out = self.act_dw(self.bn_dw(self.dw(x)))
        out = self.bn_pw(self.pw(out))
        out = self.se(out)

        if self.use_res:
            out = out + identity

        out = self.act_pw(out)
        out = self.drop(out)
        return out


# ─────────────────────────────────────────────
# HEADS
# ─────────────────────────────────────────────

class SVHead(nn.Module):
    """
    Speaker Verification Head — Early Exit at Block 2.
    Hidden dimension expanded to 512 to hit the 2.4M parameter target.
    """
    def __init__(self, in_dim=256, hidden_dim=512, embedding_dim=192, dropout_p=0.15):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU6(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward(self, x):
        x = self.pool(x).flatten(1)
        x = self.mlp(x)
        return F.normalize(x, p=2, dim=1)


class KWSHead(nn.Module):
    """
    Keyword Spotting Head — Deep Exit at Block 4.
    """
    def __init__(self, in_dim=512, num_classes=4, dropout_p=0.15):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.drop = nn.Dropout(p=dropout_p)
        self.classifier = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.classifier(x)


# ─────────────────────────────────────────────
#  COMPLETE PCOV-KWS MODEL (2.4M PARAMETERS)
# ─────────────────────────────────────────────

class PCOVKWS(nn.Module):
    """
    PCOV-KWS-Edge: Asymmetric Dual-Head DS-CNN.
    Target: ~2.4M parameters (Perfectly matches PPT presentation).
    """
    def __init__(self, num_kws_classes=4, sv_embedding_dim=192, dropout_p=0.15):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU6(inplace=True)
        )
        
        # Block 1: Shared Extractor
        self.block1 = nn.Sequential(
            DSConvBlock(64, 128, stride=1, dropout_p=dropout_p),
            DSConvBlock(128, 128, stride=1, dropout_p=dropout_p)
        )
        
        # Block 2: Shared & SV Exit (Retains 80-bin freq resolution)
        self.block2 = nn.Sequential(
            DSConvBlock(128, 256, stride=(1, 2), dropout_p=dropout_p),
            DSConvBlock(256, 256, stride=1, dropout_p=dropout_p),
            DSConvBlock(256, 256, stride=1, dropout_p=dropout_p)
        )

        self.sv_head = SVHead(in_dim=256, hidden_dim=512, embedding_dim=sv_embedding_dim, dropout_p=dropout_p)

        # Block 3: Deep KWS Path (Expanded to 4 layers for ~2.4M total params)
        self.block3 = nn.Sequential(
            DSConvBlock(256, 512, stride=(2, 2), dropout_p=dropout_p),
            DSConvBlock(512, 512, stride=1, dropout_p=dropout_p),
            DSConvBlock(512, 512, stride=1, dropout_p=dropout_p),
            DSConvBlock(512, 512, stride=1, dropout_p=dropout_p) # 🔥 Added 4th layer for max abstraction
        )
        
        # Block 4: Final KWS Abstraction
        self.block4 = nn.Sequential(
            DSConvBlock(512, 512, stride=(1, 2), dropout_p=dropout_p),
            DSConvBlock(512, 512, stride=1, dropout_p=dropout_p)
        )

        self.kws_head = KWSHead(in_dim=512, num_classes=num_kws_classes, dropout_p=dropout_p)

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        
        shared_sv = self.block2(x)  
        sv_embedding = self.sv_head(shared_sv)

        x = self.block3(shared_sv)
        shared_kws = self.block4(x)
        kws_logits = self.kws_head(shared_kws)
        
        return sv_embedding, kws_logits

if __name__ == "__main__":
    model = PCOVKWS()
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model Parameters: {total_params:,} (Target: < 3,000,000)")
    
    # Test forward pass with dummy frontend output
    dummy_input = torch.randn(8, 1, 80, 99) # [Batch, Channels, Mels, Time]
    sv_emb, kws_logits = model(dummy_input)
    
    print(f"SV Embedding Shape: {sv_emb.shape} (Expected: 8, 192)")
    print(f"KWS Logits Shape: {kws_logits.shape} (Expected: 8, 4)")

Writing model.py


******LOSS******

In [11]:
%%writefile loss.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ─────────────────────────────────────────────
# USAGE GUIDE
# SV Head  → SphereFace2Loss(sv_embedding_dim, num_speakers)
# # KWS Head → KWSLoss() using CrossEntropyLoss
# AAMSoftmax provided as alternative for SV if needed
# ─────────────────────────────────────────────


class SphereFace2Loss(nn.Module):
    """
    SphereFace2-based Binary Cosine Classification Loss.
    Pan & Huang (PCOV-KWS, 2026) — key training criterion.
    Transforms multi-class into multiple binary classifications.
    Eliminates inter-category competition between speaker classes.

    Used for: SV Head training
    Input: L2-normalized 192-dim speaker embeddings
    num_classes: number of speaker identities in training set
    """
    def __init__(self, embedding_dim, num_classes,
                 margin=0.5, scale=32.0):
        super().__init__()
        self.weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim)
        )
        nn.init.xavier_uniform_(self.weight)
        self.margin = margin
        self.scale = scale

    def forward(self, embeddings, labels):
        # Defensive L2 normalize — SVHead already does this
        # but enforced here for training safety
        embeddings = F.normalize(embeddings, p=2, dim=1)

        # Normalize class weight vectors onto hypersphere
        weight_norm = F.normalize(self.weight, p=2, dim=1)

        # Cosine similarity scores: (Batch, num_classes)
        cosine = F.linear(embeddings, weight_norm)

        # One-hot target matrix
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Apply angular margin to target class only
        cosine_m = cosine - self.margin * one_hot
        output = self.scale * cosine_m

        # Binary cross-entropy per class — SphereFace2 key trait
        return F.binary_cross_entropy_with_logits(output, one_hot)


class AAMSoftmax(nn.Module):
    """
    Additive Angular Margin Softmax — ArcFace.
    Deng et al., CVPR 2019. Industry standard for SV.
    Alternative to SphereFace2 for SV head if needed.

    Used for: SV Head (alternative to SphereFace2)
    Input: L2-normalized embeddings
    """
    def __init__(self, embedding_dim, num_classes,
                 margin=0.2, scale=30.0):
        super().__init__()
        self.weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim)
        )
        nn.init.xavier_uniform_(self.weight)
        self.margin = margin
        self.scale = scale
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings, labels):
        # Defensive normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)

        weight_norm = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, weight_norm)
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))

        # cos(theta + margin)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        return F.cross_entropy(output, labels)


class KWSLoss(nn.Module):
    """
    Keyword Spotting Loss — Standard Cross Entropy.
    Mutually exclusive 4-class classification:
    [0: Target Word, 1: Silence, 2: Unknown, 3: Hard-Negative]
    
    NOTE: Each audio clip must have exactly ONE label.
    Do NOT use this if clips can have multiple simultaneous labels.
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss()
        self.num_classes = num_classes

    def forward(self, logits, labels):
        # logits: (Batch, 4)
        # labels: (Batch,) — integer class indices
        assert logits.shape[1] == self.num_classes, \
            f"Expected {self.num_classes} classes, got {logits.shape[1]}"
        return self.criterion(logits, labels.long())



class KDLoss(nn.Module):
    """
    Knowledge Distillation Loss (Hinton et al., 2015).
    Forces the tiny Student model to mimic the soft probabilities of a massive Teacher model.
    """
    def __init__(self, temperature=3.0, alpha=0.5):
        super().__init__()
        assert temperature > 0, "Temperature must be strictly positive to prevent NaN/Inverted gradients."
        assert 0 <= alpha <= 1, "Alpha must be bounded between 0.0 and 1.0."
        
        self.T = temperature  
        self.alpha = alpha    
        self.criterion_ce = nn.CrossEntropyLoss()
        self.criterion_kl = nn.KLDivLoss(reduction='batchmean')

    
    def forward(self, student_logits, teacher_logits, labels):
        # [THE FIX] Fail-fast shape assertion
        assert student_logits.shape == teacher_logits.shape, \
            f"KWS KD Error: Student shape {student_logits.shape} must match Teacher shape {teacher_logits.shape}"
            
        # 1. Hard Loss
        hard_loss = self.criterion_ce(student_logits, labels)
        
        # 2. Soft Loss (KL Divergence)
        student_log_probs = F.log_softmax(student_logits / self.T, dim=1)
        teacher_probs = F.softmax(teacher_logits / self.T, dim=1).detach()
        
        soft_loss = self.criterion_kl(student_log_probs, teacher_probs) * (self.T ** 2)
        
        # 3. Blend them together
        return (1. - self.alpha) * hard_loss + self.alpha * soft_loss

class SVEmbeddingKDLoss(nn.Module):
    """
    Knowledge Distillation for Speaker Verification (Embeddings).
    Forces the Student's 192D embedding vector to align exactly with 
    the heavy Teacher's (ECAPA-TDNN) embedding vector.
    """
    def __init__(self, alpha=0.5):
        super().__init__()
        assert 0 <= alpha <= 1, "Alpha must be bounded between 0.0 and 1.0."
        self.alpha = alpha
        self.criterion_cos = nn.CosineEmbeddingLoss()


    def forward(self, student_emb, teacher_emb, hard_sv_loss):
        # [THE FIX] Fail-fast shape assertion
        assert student_emb.shape == teacher_emb.shape, \
            f"SV KD Error: Student shape {student_emb.shape} must match Teacher shape {teacher_emb.shape}"
            
        # Student embedding retains gradient flow
        student_emb = F.normalize(student_emb, p=2, dim=1)
        
        # Teacher embedding graph is severed.
        teacher_emb = F.normalize(teacher_emb, p=2, dim=1).detach()
        
        # Target tensor of 1s means "make these two vectors identical"
        target = torch.ones(student_emb.size(0)).to(student_emb.device)
        
        soft_loss = self.criterion_cos(student_emb, teacher_emb, target)
        
        return (1. - self.alpha) * hard_sv_loss + self.alpha * soft_loss
    

Writing loss.py


******TEACHER_LOADER******

In [12]:
%%writefile teacher_loader.py
import torch
import torch.nn as nn
import warnings


class MatchboxTeacher(nn.Module):
    """
    KWS Teacher — NVIDIA NeMo MatchboxNet3x1x64_v2.
    Pretrained on Google Speech Commands v2 (35 classes).
    Used for knowledge distillation into the student KWS head.
    Frozen at load — no gradient flow during training.

    Input:  (B, 1, 16000) or (B, 16000) — raw 16kHz waveform
    Output: (B, num_classes) — logits
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.use_real_model = False

        try:
            import nemo.collections.asr as nemo_asr
            print("🔄 Loading NVIDIA NeMo MatchboxNet...")
            self.nemo_model = nemo_asr.models.EncDecClassificationModel.from_pretrained(
                model_name="commandrecognition_en_matchboxnet3x1x64_v2"
            )
            self.nemo_model.eval()
            for p in self.nemo_model.parameters():
                p.requires_grad = False
            self.use_real_model = True
            print("✅ MatchboxNet loaded and frozen.")
        except (ImportError, Exception) as e:
            warnings.warn(f"NeMo unavailable ({type(e).__name__}). Run: pip install nemo_toolkit[asr]")
            self.fallback = nn.Linear(16000, num_classes)

    def forward(self, x):
        # Accept (B,1,16000) from train.py or (B,16000) directly
        if x.dim() == 3:
            x = x.squeeze(1)

        if self.use_real_model:
            # Force model to same device as input — NeMo does not auto-track device
            self.nemo_model = self.nemo_model.to(x.device)
            lens = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)
            logits, _ = self.nemo_model(audio_signal=x, audio_signal_length=lens)
            return logits

        return self.fallback(x)


class ECAPATeacher(nn.Module):
    """
    SV Teacher — SpeechBrain ECAPA-TDNN pretrained on VoxCeleb.
    Outputs 192-dim speaker embeddings.
    Used for knowledge distillation into the student SV head.
    Frozen at load — no gradient flow during training.

    Input:  (B, 1, 16000) or (B, 16000) — raw 16kHz waveform
    Output: (B, embedding_dim) — L2-normalized speaker embedding
    """
    def __init__(self, embedding_dim=192):
        super().__init__()
        self.use_real_model = False
        self.proj = None

        try:
            from speechbrain.inference.speaker import EncoderClassifier
            print("🔄 Loading SpeechBrain ECAPA-TDNN...")
            self.sb_model = EncoderClassifier.from_hparams(
                source="speechbrain/spkrec-ecapa-voxceleb",
                savedir="/kaggle/working/tmp_sb_models",
                run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"}
            )
            # Verify output dim — ECAPA outputs 192 natively but guard anyway
            with torch.no_grad():
                test = self.sb_model.encode_batch(torch.zeros(1, 16000)).squeeze(1)
            if test.shape[-1] != embedding_dim:
                self.proj = nn.Linear(test.shape[-1], embedding_dim)
                print(f"⚠️  Projection added: {test.shape[-1]}→{embedding_dim}")

            for p in self.sb_model.parameters():
                p.requires_grad = False
            self.use_real_model = True
            print("✅ ECAPA-TDNN loaded and frozen.")
        except (ImportError, Exception) as e:
            warnings.warn(f"SpeechBrain unavailable ({type(e).__name__}). Run: pip install speechbrain")
            self.fallback = nn.Linear(16000, embedding_dim)

    def forward(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)

        if self.use_real_model:
            # Sync SpeechBrain internals to match inbound tensor device
            self.sb_model.device = x.device
            self.sb_model.mods = self.sb_model.mods.to(x.device)
            emb = self.sb_model.encode_batch(x).squeeze(1)
            if self.proj is not None:
                emb = self.proj(emb)
            return emb

        return self.fallback(x)


Writing teacher_loader.py


In [13]:
import subprocess
subprocess.run(["pip", "install", "speechbrain", "-q"], check=True)


CompletedProcess(args=['pip', 'install', 'speechbrain', '-q'], returncode=0)

******PRECOMPUTE EMBEDDINGS******

In [14]:
%%writefile precompute_embedding.py
import os
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import soundfile as sf
import torchaudio.functional as F_audio
from teacher_loader import ECAPATeacher

TRAIN_MANIFEST = "/kaggle/working/pcov-kws-manifests/train_manifest.json"
VAL_MANIFEST   = "/kaggle/working/pcov-kws-manifests/val_manifest.json"
OUTPUT_PATH    = "/kaggle/working/sv_embeddings.pt"
BATCH_SIZE     = 256
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class RawAudioDataset(Dataset):
    def __init__(self, manifest):
        self.manifest   = manifest
        self.target_len = 16000

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        raw_path = self.manifest[idx]['path']
        path = str(raw_path).replace('\n', '').replace('\r', '').strip()
        try:
            wav, sr  = sf.read(path)
            waveform = torch.from_numpy(wav).float()
            if waveform.ndim == 1:
                waveform = waveform.unsqueeze(0)
            else:
                waveform = waveform.t()
            if sr != 16000:
                waveform = F_audio.resample(waveform, sr, 16000)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            if waveform.shape[1] > self.target_len:
                waveform = waveform[:, :self.target_len]
            elif waveform.shape[1] < self.target_len:
                waveform = torch.nn.functional.pad(
                    waveform, (0, self.target_len - waveform.shape[1])
                )
        except Exception as e:
            print(f"⚠️ Failed: {path} — {e}")
            waveform = torch.zeros(1, self.target_len)

        # 🛡️ THE VACCINE: Catch NaN, Inf, or ultra-low RMS instantly!
        if torch.isnan(waveform).any() or torch.isinf(waveform).any() or waveform.pow(2).mean().sqrt() < 1e-8:
            waveform = torch.zeros(1, self.target_len)

        return waveform, idx

def main():
    print("=" * 60)
    print("🚀 ECAPA-TDNN EMBEDDING PRE-COMPUTATION")
    print("=" * 60)
    print(f"Device: {DEVICE}")

    with open(TRAIN_MANIFEST) as f:
        train_manifest = json.load(f)
    with open(VAL_MANIFEST) as f:
        val_manifest = json.load(f)

    full_manifest = train_manifest + val_manifest
    total         = len(full_manifest)
    print(f"Total files: {total:,}")

    sv_teacher = ECAPATeacher(embedding_dim=192).to(DEVICE)
    for p in sv_teacher.parameters():
        p.requires_grad = False
    sv_teacher.eval()
    if sv_teacher.use_real_model:
        sv_teacher.sb_model.device = DEVICE
        sv_teacher.sb_model.mods   = sv_teacher.sb_model.mods.to(DEVICE)
    print("✅ ECAPA-TDNN loaded and frozen")

    all_embeddings = torch.zeros(total, 192, dtype=torch.float32)

    dataset = RawAudioDataset(full_manifest)
    loader  = DataLoader(
        dataset, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True
    )

    print(f"\nProcessing {total:,} files...")
    with torch.no_grad():
        for waveforms, indices in tqdm(loader, desc="Pre-computing"):
            waveforms  = waveforms.to(DEVICE)
            embeddings = sv_teacher(waveforms).cpu().float()
            
            # 🛡️ THE SECOND VACCINE: Catch any NaN embeddings
            nan_mask = torch.isnan(embeddings).any(dim=1)
            if nan_mask.any():
                safe_vec = torch.zeros(192, dtype=torch.float32)
                safe_vec[0] = 1.0          # unit vector — valid on hypersphere
                embeddings[nan_mask] = safe_vec
                print(f"\n⚠️ Fixed {nan_mask.sum().item()} NaN embeddings in batch!")

            all_embeddings[indices] = embeddings

    torch.save({
        'train_count': len(train_manifest),
        'val_count':   len(val_manifest),
        'embeddings':  all_embeddings,
    }, OUTPUT_PATH)

    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"\n✅ Saved {total:,} embeddings → {OUTPUT_PATH}")
    print(f"   Size: {size_mb:.1f} MB | Shape: {all_embeddings.shape}")
    print("✅ Done. Now run train.py")

if __name__ == "__main__":
    main()

Writing precompute_embedding.py


In [15]:
# %run precompute_embedding.py

🚀 ECAPA-TDNN EMBEDDING PRE-COMPUTATION
Device: cuda
Total files: 144,000
🔄 Loading SpeechBrain ECAPA-TDNN...


hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


✅ ECAPA-TDNN loaded and frozen.
✅ ECAPA-TDNN loaded and frozen

Processing 144,000 files...


Pre-computing: 100%|██████████| 563/563 [07:16<00:00,  1.29it/s]


✅ Saved 144,000 embeddings → /kaggle/working/sv_embeddings.pt
   Size: 105.5 MB | Shape: torch.Size([144000, 192])
✅ Done. Now run train.py


In [ ]:
%%writefile train.py
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report

from frontend import PCENFrontend
from model import PCOVKWS
from dataset import PCOVMultiTaskDataset
from loss import SVEmbeddingKDLoss

# ─────────────────────────────────────────────
# KAGGLE PATHS
# ─────────────────────────────────────────────
TRAIN_MANIFEST   = "/kaggle/working/pcov-kws-manifests/train_manifest.json"
VAL_MANIFEST     = "/kaggle/working/pcov-kws-manifests/val_manifest.json"
EMBEDDINGS_PATH  = "/kaggle/working/sv_embeddings.pt"
MUSAN_DIR        = "/kaggle/input/datasets/nhattruongdev/musan-noise"
RIR_DIR          = "/kaggle/input/datasets/nhattruongdev/rirs-noises"

CHECKPOINT_DIR   = "/kaggle/working/checkpoints"
BACKDOOR_PATH    = "/kaggle/working/checkpoints/last_checkpoint.pth"

if os.path.exists(BACKDOOR_PATH):
    RESUME_PATH = BACKDOOR_PATH
    print(f"🎯 Found local checkpoint weights at {BACKDOOR_PATH}!")
else:
    RESUME_PATH = f"{CHECKPOINT_DIR}/last_checkpoint.pth"

BEST_PATH    = f"{CHECKPOINT_DIR}/pcov_kws_best.pth"
NUM_EPOCHS   = 50
BATCH_SIZE   = 64
SV_FREEZE_EPOCHS = 15  # SV head frozen for first 15 epochs


# ─────────────────────────────────────────────
# 1. PCGRAD — GRADIENT SURGERY OPTIMIZER (FIXED)
# ─────────────────────────────────────────────
class PCGradOptimizer:
    def __init__(self, optimizer):
        self.optimizer = optimizer

    def zero_grad(self):
        self.optimizer.zero_grad()

    def step(self, scaler=None):
        if scaler is not None:
            scaler.step(self.optimizer)
            scaler.update()
        else:
            self.optimizer.step()

    def pc_backward(self, losses, scaler=None):
        num_tasks  = len(losses)
        task_grads = []

        for i in range(num_tasks):
            self.optimizer.zero_grad()
            retain = (i < num_tasks - 1)

            if scaler is not None:
                # Scale and backward — do NOT manually unscale here
                scaler.scale(losses[i]).backward(retain_graph=retain)
                grads = [
                    p.grad.detach().clone() if p.grad is not None
                    else torch.zeros_like(p)
                    for p in self.optimizer.param_groups[0]['params']
                ]
            else:
                losses[i].backward(retain_graph=retain)
                grads = [
                    p.grad.detach().clone() if p.grad is not None
                    else torch.zeros_like(p)
                    for p in self.optimizer.param_groups[0]['params']
                ]
            task_grads.append(grads)

        self.optimizer.zero_grad()

        # Gradient surgery — operates on scaled grads (consistent scale, surgery still valid)
        for i in range(num_tasks):
            for j in range(num_tasks):
                if i == j:
                    continue
                dot = sum(
                    torch.sum(g_i * g_j)
                    for g_i, g_j in zip(task_grads[i], task_grads[j])
                )
                if dot < 0:
                    norm_sq = sum(
                        torch.sum(g_j * g_j) for g_j in task_grads[j]
                    ) + 1e-8
                    task_grads[i] = [
                        g_i - (dot / norm_sq) * g_j
                        for g_i, g_j in zip(task_grads[i], task_grads[j])
                    ]

        # Write summed gradients back — still scaled at this point
        for param_idx, p in enumerate(self.optimizer.param_groups[0]['params']):
            p.grad = sum(task_grads[i][param_idx] for i in range(num_tasks))


# ─────────────────────────────────────────────
# 2. DATASET WRAPPER
# ─────────────────────────────────────────────
class EmbeddingDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, embeddings, offset=0):
        self.base   = base_dataset
        self.emb    = embeddings
        self.offset = offset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        waveform, kws_label, _ = self.base[idx]
        sv_emb = self.emb[self.offset + idx]
        return waveform, kws_label, sv_emb


# ─────────────────────────────────────────────
# 3. TRAINING EPOCH (FIXED)
# ─────────────────────────────────────────────
def train_epoch(model, frontend, dataloader,
                kws_hard_criterion, sv_kd_criterion,
                pc_optimizer, scaler, device, scheduler):

    model.train()
    frontend.train()

    total_kws_loss, total_sv_loss = 0.0, 0.0
    correct_kws, total_kws = 0, 0
    target_ones = torch.ones(BATCH_SIZE, device=device)

    for waveform, kws_label, t_sv_emb in tqdm(dataloader, desc="Training", leave=False):
        waveform  = waveform.to(device)
        kws_label = kws_label.to(device)
        t_sv_emb  = t_sv_emb.to(device)

        if waveform.size(0) != target_ones.size(0):
            target_ones = torch.ones(waveform.size(0), device=device)

        pc_optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=scaler is not None):
            features            = frontend(waveform)
            s_sv_emb, s_kws_out = model(features)

        if kws_hard_criterion.weight is not None:
            kws_hard_criterion.weight = kws_hard_criterion.weight.to(s_kws_out.dtype)

        loss_kws = kws_hard_criterion(s_kws_out, kws_label)
        loss_sv  = sv_kd_criterion(
            s_sv_emb,
            t_sv_emb.to(s_sv_emb.dtype).detach(),
            hard_sv_loss=torch.tensor(0.0, device=device)
        )

        # 🛡️ NaN shield — no scheduler.step() on skip
        if torch.isnan(loss_kws) or torch.isnan(loss_sv) or \
                torch.isinf(loss_kws) or torch.isinf(loss_sv):
            print("\n⚠️ NaN/Inf loss detected — skipping batch...")
            pc_optimizer.zero_grad()
            continue

        # pc_backward writes scaled grads into p.grad
        pc_optimizer.pc_backward([loss_kws, loss_sv], scaler=scaler)

        # Unscale once here — converts scaled grads to true grads for clip
        if scaler is not None:
            scaler.unscale_(pc_optimizer.optimizer)

        # Clip on true gradient magnitudes
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    
        torch.nn.utils.clip_grad_norm_(frontend.parameters(), max_norm=1.0)

        # scaler.step() checks for inf/nan, skips optimizer if found
        pc_optimizer.step(scaler=scaler)
        scheduler.step()

        total_kws_loss += loss_kws.item()
        total_sv_loss  += loss_sv.item()
        preds           = torch.argmax(s_kws_out, dim=1)
        correct_kws    += (preds == kws_label).sum().item()
        total_kws      += kws_label.size(0)

    n = len(dataloader)
    return total_kws_loss / n, total_sv_loss / n, 100.0 * correct_kws / total_kws

# ─────────────────────────────────────────────
# 4. VALIDATION EPOCH
# ─────────────────────────────────────────────
def validate_epoch(model, frontend, dataloader,
                   kws_hard_criterion, sv_kd_criterion,  # ← renamed
                   device):

    model.eval()
    frontend.eval()

    total_kws_loss, total_sv_loss = 0.0, 0.0
    correct_kws, total_kws = 0, 0
    target_ones = torch.ones(BATCH_SIZE, device=device)

    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for waveform, kws_label, t_sv_emb in tqdm(dataloader, desc="Validating", leave=False):
            waveform  = waveform.to(device)
            kws_label = kws_label.to(device)
            t_sv_emb  = t_sv_emb.to(device)

            if waveform.size(0) != target_ones.size(0):
                target_ones = torch.ones(waveform.size(0), device=device)

            with torch.amp.autocast('cuda'):
                features            = frontend(waveform)
                s_sv_emb, s_kws_out = model(features)

            if kws_hard_criterion.weight is not None:
                kws_hard_criterion.weight = kws_hard_criterion.weight.to(s_kws_out.dtype)

            loss_kws = kws_hard_criterion(s_kws_out, kws_label)
            loss_sv  = sv_kd_criterion(                      # ← SVEmbeddingKDLoss
                s_sv_emb,
                t_sv_emb.to(s_sv_emb.dtype),
                hard_sv_loss=torch.tensor(0.0, device=device)
            )

            total_kws_loss += loss_kws.item()
            total_sv_loss  += loss_sv.item()
            preds           = torch.argmax(s_kws_out, dim=1)
            correct_kws    += (preds == kws_label).sum().item()
            total_kws      += kws_label.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(kws_label.cpu().numpy())

    n = len(dataloader)
    target_names = ['Wake Word', 'Silence', 'Unknown', 'Hard-Negative']
    report = classification_report(
        all_labels, all_preds,
        target_names=target_names,
        zero_division=0
    )

    return total_kws_loss / n, total_sv_loss / n, 100.0 * correct_kws / total_kws, report


# ─────────────────────────────────────────────
# 5. MAIN
# ─────────────────────────────────────────────
def main():
    print("=" * 60)
    print("🚀 PCOV-KWS PHASE 2 — 404NOT-FOUND TRAINING PIPELINE")
    print("=" * 60)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔥 Hardware: {device}")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    with open(TRAIN_MANIFEST) as f:
        train_manifest = json.load(f)
    with open(VAL_MANIFEST) as f:
        val_manifest = json.load(f)
    print(f"📂 Train: {len(train_manifest):,} | Val: {len(val_manifest):,} samples")

    print(f"📦 Loading precomputed SV embeddings from {EMBEDDINGS_PATH}...")
    emb_data    = torch.load(EMBEDDINGS_PATH, map_location='cpu')
    embeddings  = emb_data['embeddings']
    train_count = emb_data['train_count']

    train_base = PCOVMultiTaskDataset(
        data_manifest=train_manifest,
        musan_dir=MUSAN_DIR,
        rir_dir=RIR_DIR
    )
    val_base = PCOVMultiTaskDataset(
        data_manifest=val_manifest,
        musan_dir=None,
        rir_dir=None
    )

    train_dataset = EmbeddingDataset(train_base, embeddings, offset=0)
    val_dataset   = EmbeddingDataset(val_base,   embeddings, offset=train_count)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True
    )

    frontend = PCENFrontend().to(device)
    model    = PCOVKWS(num_kws_classes=4, sv_embedding_dim=192).to(device)

    # ── SV head freeze helper ──
    def set_sv_frozen(frozen: bool):
        for p in model.sv_head.parameters():
            p.requires_grad = not frozen
        print(f"{'🔒 SV head FROZEN' if frozen else '🔓 SV head UNFROZEN'}")

    set_sv_frozen(frozen=True)  # freeze at init

    print("📊 Computing class weights dynamically from train_manifest...")
    label_key = 'label' if 'label' in train_manifest[0] else 'kws_label'
    counts = [0, 0, 0, 0]
    for item in train_manifest:
        counts[item[label_key]] += 1
    counts = [max(c, 1) for c in counts]
    print(f"Detected Class Counts: {counts}")

    class_counts  = torch.tensor(counts, dtype=torch.float, device=device)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * 4.0

    kws_hard_criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)
    sv_kd_criterion    = SVEmbeddingKDLoss(alpha=0.5).to(device)


    base_optimizer = optim.AdamW(
        list(model.parameters()) + list(frontend.parameters()),
        lr=5e-4, weight_decay=1e-4
    )

    pc_optimizer = PCGradOptimizer(base_optimizer) 
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        base_optimizer,
        max_lr=5e-4,
        epochs=NUM_EPOCHS,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy='cos'
    )


    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    start_epoch   = 0
    best_val_loss = float('inf')

    if os.path.exists(RESUME_PATH):
        print(f"🔄 Resuming from {RESUME_PATH}")
        ckpt = torch.load(RESUME_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        frontend.load_state_dict(ckpt['frontend_state_dict'])
        base_optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            try:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
                print("✅ Scheduler state restored")
            except Exception as e:
                print(f"⚠️ Scheduler state mismatch — starting scheduler fresh: {e}")
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        start_epoch   = ckpt['epoch'] + 1
        best_val_loss = ckpt.get('val_loss', float('inf'))
        print(f"⏩ Resuming from Epoch {start_epoch + 1}/{NUM_EPOCHS}")

        # Restore freeze state on resume
        if start_epoch < SV_FREEZE_EPOCHS:
            set_sv_frozen(frozen=True)
        else:
            set_sv_frozen(frozen=False)
    else:
        print("🆕 No checkpoint — starting fresh")

    print(f"\n✅ Pipeline ready. Training on {device}...\n")

    for epoch in range(start_epoch, NUM_EPOCHS):

        # ── SV freeze/unfreeze boundary ──
        if epoch == SV_FREEZE_EPOCHS:
            set_sv_frozen(frozen=False)

        print(f"--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")

        train_kws_l, train_sv_l, train_acc = train_epoch(
            model, frontend, train_loader,
            kws_hard_criterion, sv_kd_criterion,
            pc_optimizer, scaler, device, scheduler
        )

        val_kws_l, val_sv_l, val_acc, val_report = validate_epoch(
            model, frontend, val_loader,
            kws_hard_criterion, sv_kd_criterion,
            device
        )

        combined_val_loss = 0.5 * val_kws_l + 0.5 * val_sv_l

        print(f"Train | KWS: {train_kws_l:.4f} | SV: {train_sv_l:.4f} | Acc: {train_acc:.2f}%")
        print(f"Val   | KWS: {val_kws_l:.4f}   | SV: {val_sv_l:.4f}   | Acc: {val_acc:.2f}%")
        print(f"LR    | {scheduler.get_last_lr()[0]:.6f}")

        print("\n" + "-" * 50)
        print("📊 VALIDATION CLASSIFICATION REPORT")
        print("-" * 50)
        print(val_report)
        print("-" * 50 + "\n")

        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'frontend_state_dict':  frontend.state_dict(),
            'optimizer_state_dict': base_optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict':    scaler.state_dict() if scaler else None,
            'val_loss':             best_val_loss,
        }, RESUME_PATH)

        if combined_val_loss < best_val_loss:
            best_val_loss = combined_val_loss
            torch.save({
                'epoch':               epoch,
                'model_state_dict':    model.state_dict(),
                'frontend_state_dict': frontend.state_dict(),
                'val_loss':            best_val_loss,
            }, BEST_PATH)
            print(f"🌟 New best saved! Val Loss: {best_val_loss:.4f}")

    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETE")
    print(f"📂 Best model: {BEST_PATH}")
    print("=" * 60)

if __name__ == "__main__":
    main()


In [17]:
%run train.py

🚀 PCOV-KWS PHASE 2 — 404NOT-FOUND TRAINING PIPELINE
🔥 Hardware: cuda
📂 Train: 115,093 | Val: 28,907 samples
📦 Loading precomputed SV embeddings from /kaggle/working/sv_embeddings.pt...
📥 Caching 2000 RIR files into RAM...


📥 Caching 2000 MUSAN files into RAM...


🔒 SV head FROZEN
📊 Computing class weights dynamically from train_manifest...
Detected Class Counts: [35120, 8000, 60000, 11973]
🆕 No checkpoint — starting fresh

✅ Pipeline ready. Training on cuda...

--- Epoch 1/50 ---


Training:   0%|          | 0/1799 [00:00<?, ?it/s]/kaggle/working/train.py:184: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


KeyboardInterrupt: 